In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import requests
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from urllib.parse import urljoin

print("Libraries imported successfully")

Libraries imported successfully


In [5]:
url = "http://brain.bio.msu.ru/eeg_schizophrenia.htm"
save_dir = "/content/eeg_schizophrenia_dataset"

os.makedirs(save_dir, exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

download_links = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    full_url = urljoin(url, href)

    if full_url.lower().endswith(".zip"):
        download_links.append(full_url)

print("Total zip files found:", len(download_links))
print("Download links:")
for link in download_links:
    print(link)

Total zip files found: 2
Download links:
http://brain.bio.msu.ru/eeg_data/schizophrenia/norm.zip
http://brain.bio.msu.ru/eeg_data/schizophrenia/sch.zip


In [6]:
for file_url in download_links:
    filename = os.path.basename(file_url)
    filepath = os.path.join(save_dir, filename)

    if os.path.exists(filepath):
        print("Already downloaded:", filename)
        continue

    r = requests.get(file_url, headers=headers)

    with open(filepath, "wb") as f:
        f.write(r.content)

    print("Downloaded:", filename)

Downloaded: norm.zip
Downloaded: sch.zip


In [7]:
for file in os.listdir(save_dir):
    if file.endswith(".zip"):
        zip_path = os.path.join(save_dir, file)

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(save_dir)

        print("Extracted:", file)

Extracted: norm.zip
Extracted: sch.zip


In [8]:
eea_files = []

for root, dirs, files in os.walk(save_dir):
    for file in files:
        if file.lower().endswith(".eea"):
            eea_files.append(os.path.join(root, file))

print("Total .eea files found:", len(eea_files))

for f in eea_files[:10]:
    print(f)

Total .eea files found: 84
/content/eeg_schizophrenia_dataset/S155W1.eea
/content/eeg_schizophrenia_dataset/509w1.eea
/content/eeg_schizophrenia_dataset/s20w1.eea
/content/eeg_schizophrenia_dataset/S179W1.eea
/content/eeg_schizophrenia_dataset/429w1.eea
/content/eeg_schizophrenia_dataset/221w.eea
/content/eeg_schizophrenia_dataset/192w.eea
/content/eeg_schizophrenia_dataset/S165W1.eea
/content/eeg_schizophrenia_dataset/382w1.eea
/content/eeg_schizophrenia_dataset/585w1.eea


In [9]:
# Sort files for consistency
eea_files = sorted(eea_files)

print("Total files:", len(eea_files))
print("First 10 files:")
for f in eea_files[:10]:
    print(os.path.basename(f))

Total files: 84
First 10 files:
022w1.eea
088w1.eea
103w.eea
113w1.eea
155w1.eea
156w1.eea
192w.eea
219w1.eea
221w.eea
249w1.eea


In [10]:
sample_file = eea_files[0]

sample_data = np.loadtxt(sample_file)

print("Sample file:", os.path.basename(sample_file))
print("Raw shape:", sample_data.shape)
print(sample_data[:5])

Sample file: 022w1.eea
Raw shape: (122880,)
[108.01 208.82 388.84 446.45 446.45]


In [11]:
X_subjects = []
subject_ids = []
labels = []

for file_path in eea_files:
    file_name = os.path.basename(file_path)

    data = np.loadtxt(file_path)

    # Convert 1D data into 16 channels × 7680 samples
    data = data.reshape(16, 7680)

    X_subjects.append(data)
    subject_ids.append(file_name)

    # Label rule:
    # Files starting with S/s = schizophrenia
    # Other files = healthy control
    if file_name.lower().startswith("s"):
        labels.append(1)
    else:
        labels.append(0)

X_subjects = np.array(X_subjects)
labels = np.array(labels)

print("X_subjects shape:", X_subjects.shape)
print("Labels shape:", labels.shape)

print("Schizophrenia subjects:", np.sum(labels == 1))
print("Healthy subjects:", np.sum(labels == 0))

print("First 10 subject IDs and labels:")
for i in range(10):
    print(subject_ids[i], labels[i])

X_subjects shape: (84, 16, 7680)
Labels shape: (84,)
Schizophrenia subjects: 43
Healthy subjects: 41
First 10 subject IDs and labels:
022w1.eea 0
088w1.eea 0
103w.eea 0
113w1.eea 0
155w1.eea 0
156w1.eea 0
192w.eea 0
219w1.eea 0
221w.eea 0
249w1.eea 0


In [12]:
sch_files = [name for name, label in zip(subject_ids, labels) if label == 1]
healthy_files = [name for name, label in zip(subject_ids, labels) if label == 0]

print("Schizophrenia files:", len(sch_files))
print(sch_files)

print("\nHealthy files:", len(healthy_files))
print(healthy_files[:10])

Schizophrenia files: 43
['S084-1W.eea', 'S10W1.eea', 'S153W1.eea', 'S154W1.eea', 'S155W1.eea', 'S163W1.eea', 'S164W1.eea', 'S165W1.eea', 'S167W1.eea', 'S169W.eea', 'S174W1.eea', 'S177W1.eea', 'S179W1.eea', 'S182W1.eea', 'S18W1.eea', 'S196W1.eea', 'S26W1.eea', 'S31W.eea', 'S42W1.eea', 'S47W1.eea', 'S50W.eea', 'S55W1.eea', 'S59LW.eea', 'S60W.eea', 'S72W1.eea', 'S78W.eea', 'S85W1.eea', 's083w1.eea', 's12w1.eea', 's152w1.eea', 's157w1.eea', 's158w1.eea', 's170w1.eea', 's173w1.eea', 's176w1.eea', 's178w1.eea', 's20w1.eea', 's27w1.eea', 's351w.eea', 's425w1.eea', 's43w1.eea', 's53w1.eea', 's94w1.eea']

Healthy files: 41
['022w1.eea', '088w1.eea', '103w.eea', '113w1.eea', '155w1.eea', '156w1.eea', '192w.eea', '219w1.eea', '221w.eea', '249w1.eea']


In [13]:
fs = 128
window_sec = 5
window_size = fs * window_sec   # 128 × 5 = 640 samples

X_epochs = []
y_epochs = []
groups = []

for subject_index, subject_data in enumerate(X_subjects):
    label = labels[subject_index]

    total_samples = subject_data.shape[1]
    num_epochs = total_samples // window_size

    for epoch_index in range(num_epochs):
        start = epoch_index * window_size
        end = start + window_size

        epoch = subject_data[:, start:end]

        X_epochs.append(epoch)
        y_epochs.append(label)
        groups.append(subject_index)

X_epochs = np.array(X_epochs)
y_epochs = np.array(y_epochs)
groups = np.array(groups)

print("X_epochs shape:", X_epochs.shape)
print("y_epochs shape:", y_epochs.shape)
print("groups shape:", groups.shape)
print("Epochs per subject:", num_epochs)

print("Schizophrenia epochs:", np.sum(y_epochs == 1))
print("Healthy epochs:", np.sum(y_epochs == 0))

X_epochs shape: (1008, 16, 640)
y_epochs shape: (1008,)
groups shape: (1008,)
Epochs per subject: 12
Schizophrenia epochs: 516
Healthy epochs: 492


In [15]:
X_lstm = np.transpose(X_epochs, (0, 2, 1))

print("Original EEG epoch shape:", X_epochs.shape)
print("LSTM input shape:", X_lstm.shape)
print("Labels shape:", y_epochs.shape)
print("Groups shape:", groups.shape)

Original EEG epoch shape: (1008, 16, 640)
LSTM input shape: (1008, 640, 16)
Labels shape: (1008,)
Groups shape: (1008,)


In [16]:
print("Final Data Summary")
print("------------------")
print("Total subjects:", X_subjects.shape[0])
print("EEG channels:", X_subjects.shape[1])
print("Samples per subject:", X_subjects.shape[2])
print("Sampling frequency:", fs, "Hz")
print("Window length:", window_sec, "seconds")
print("Samples per epoch:", window_size)
print("Total epochs:", X_epochs.shape[0])
print("Original epoch shape:", X_epochs.shape)
print("LSTM input shape:", X_lstm.shape)
print("Labels shape:", y_epochs.shape)
print("Groups shape:", groups.shape)
print("Schizophrenia subjects:", np.sum(labels == 1))
print("Healthy subjects:", np.sum(labels == 0))
print("Schizophrenia epochs:", np.sum(y_epochs == 1))
print("Healthy epochs:", np.sum(y_epochs == 0))

Final Data Summary
------------------
Total subjects: 84
EEG channels: 16
Samples per subject: 7680
Sampling frequency: 128 Hz
Window length: 5 seconds
Samples per epoch: 640
Total epochs: 1008
Original epoch shape: (1008, 16, 640)
LSTM input shape: (1008, 640, 16)
Labels shape: (1008,)
Groups shape: (1008,)
Schizophrenia subjects: 43
Healthy subjects: 41
Schizophrenia epochs: 516
Healthy epochs: 492
